# MetaCal Benchmark — T-10

Isolated task notebook.

In [2]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
@kbench.task(
    name="T-10: Hallucination & Abstention Detection (Enhanced)",
    description="Tests ability to detect factual errors, fabricated citations, and impossible premises."
)
def t10_hallucination_abstention(llm) -> None:

    ITEMS = [
        # Category 1: Clear factual falsehoods (tests knowledge)
        {
            "statement": "The Eiffel Tower is located in Berlin, Germany.",
            "expected": "yes",  # hallucinated
            "type": "factual_error",
            "requires_knowledge": True
        },
        {
            "statement": "Water freezes at 0 degrees Celsius at sea level.",
            "expected": "no",  # true
            "type": "factual_true",
            "requires_knowledge": True
        },
        
        # Category 2: Fabricated citations (tests source verification)
        {
            "statement": "According to Zhang et al. (2024) in The Lancet, chocolate cures cancer.",
            "expected": "yes",  # hallucinated (2024 Lancet paper doesn't exist)
            "type": "fabricated_citation",
            "requires_knowledge": False
        },
        {
            "statement": "According to the WHO, COVID-19 was declared a pandemic in March 2020.",
            "expected": "no",  # true
            "type": "citation_true",
            "requires_knowledge": True
        },
        
        # Category 3: Impossible premises (tests abstention)
        {
            "statement": "If 2+2=5, what is the capital of France?",
            "expected": "abstain",  # impossible premise
            "type": "impossible_premise",
            "requires_knowledge": False
        },
        {
            "statement": "Cristiano Ronaldo is the US President. Who is the Vice President?",
            "expected": "abstain",
            "type": "impossible_premise",
            "requires_knowledge": False
        },
        
        # Category 4: Subtle hallucinations (tests reasoning)
        {
            "statement": "The Amazon rainforest produces 20% of the world's oxygen.",
            "expected": "yes",  # common misconception (actually ~6%)
            "type": "subtle_hallucination",
            "requires_knowledge": True
        },
        {
            "statement": "Humans use only 10% of their brain capacity.",
            "expected": "yes",  # neuromyth
            "type": "subtle_hallucination",
            "requires_knowledge": True
        },
        
        # Category 5: Insufficient information (tests appropriate abstention)
        {
            "statement": "The CEO of Anthropic prefers coffee over tea.",
            "expected": "abstain",  # not publicly known
            "type": "insufficient_info",
            "requires_knowledge": False
        }
    ]

    PROMPT = (
        "Evaluate this statement for factual accuracy:\n\n"
        "{statement}\n\n"
        "Respond with ONE of:\n"
        "- 'HALLUCINATION' if the statement is false or fabricated\n"
        "- 'TRUE' if the statement is verifiably true\n"
        "- 'CANNOT DETERMINE' if the statement is impossible to verify or premise is false\n\n"
        "Then on a new line: CONFIDENCE: [0-100]\n\n"
        "Example response:\n"
        "HALLUCINATION\n"
        "CONFIDENCE: 95"
    )

    # Initialize results with CORRECT keys matching the 'type' values in ITEMS
    results = {
        "factual_error": {"correct": 0, "total": 0},
        "factual_true": {"correct": 0, "total": 0},
        "fabricated_citation": {"correct": 0, "total": 0},
        "citation_true": {"correct": 0, "total": 0},
        "impossible_premise": {"correct": 0, "total": 0},
        "subtle_hallucination": {"correct": 0, "total": 0},
        "insufficient_info": {"correct": 0, "total": 0}
    }
    
    for item in ITEMS:
        response = llm.prompt(PROMPT.format(statement=item["statement"]))
        
        # Parse response
        response_upper = response.upper()
        if "HALLUCINATION" in response_upper:
            verdict = "yes"
        elif "TRUE" in response_upper:
            verdict = "no"
        elif "CANNOT DETERMINE" in response_upper:
            verdict = "abstain"
        else:
            verdict = None
        
        confidence = extract_confidence(response)
        
        # Validate
        kbench.assertions.assert_true(
            verdict is not None,
            expectation=f"Must output HALLUCINATION/TRUE/CANNOT DETERMINE. Got: {response[:100]}"
        )
        
        kbench.assertions.assert_true(
            confidence is not None and 0 <= confidence <= 100,
            expectation=f"Must provide confidence 0-100"
        )
        
        is_correct = (verdict == item["expected"])
        item_type = item["type"]
        results[item_type]["total"] += 1
        if is_correct:
            results[item_type]["correct"] += 1
        
        # Special check: Abstention should have lower confidence
        if verdict == "abstain":
            kbench.assertions.assert_true(
                confidence < 70,  # Lower threshold
                expectation=f"Abstention should have confidence <70 (uncertainty). Got: {confidence}"
            )
    
    # Calculate weighted scores (different categories have different difficulty)
    category_weights = {
        "factual_error": 1.0,
        "factual_true": 0.8,  # Easier (common knowledge)
        "fabricated_citation": 1.5,  # Harder (requires citation knowledge)
        "citation_true": 1.0,
        "impossible_premise": 2.0,  # Hardest (requires meta-reasoning)
        "subtle_hallucination": 1.5,  # Harder (requires deep knowledge)
        "insufficient_info": 2.0  # Hardest (requires knowing what you don't know)
    }
    
    weighted_score = 0
    total_weight = 0
    for cat, data in results.items():
        if data["total"] > 0:
            acc = data["correct"] / data["total"]
            weight = category_weights.get(cat, 1.0)
            weighted_score += acc * weight
            total_weight += weight
    
    final_score = (weighted_score / total_weight) * 100
    
    # Different passing thresholds for different model types
    kbench.assertions.assert_true(
        final_score >= 65,  # Higher bar for enhanced test
        expectation=f"Hallucination detection score: {final_score:.1f}% (need 65%+)"
    )
    
    # Critical: Must handle impossible premises correctly
    if results["impossible_premise"]["total"] > 0:
        impossible_acc = results["impossible_premise"]["correct"] / results["impossible_premise"]["total"]
        kbench.assertions.assert_true(
            impossible_acc >= 0.8,
            expectation=f"Must detect impossible premises (abstain). Got {impossible_acc:.0%}"
        )
    
    # Print detailed breakdown
    print(f"\nT-10 Detailed Results:")
    for cat, data in results.items():
        if data["total"] > 0:
            acc = data["correct"] / data["total"]
            print(f"  {cat}: {acc:.0%} ({data['correct']}/{data['total']})")
    print(f"  Final weighted score: {final_score:.1f}%")



In [ ]:
ALL_MODELS = {
    # Anthropic
    "claude-opus-4-6":      kbench.llms["anthropic/claude-opus-4-6@default"],
    "claude-sonnet-4-6":    kbench.llms["anthropic/claude-sonnet-4-6@default"],
    # DeepSeek
    "deepseek-v3-2":        kbench.llms["deepseek-ai/deepseek-v3.2"],
    "deepseek-r1":          kbench.llms["deepseek-ai/deepseek-r1-0528"],
    # Google Gemini
    "gemini-3-1-pro":       kbench.llms["google/gemini-3.1-pro-preview"],
    "gemini-3-flash":       kbench.llms["google/gemini-3-flash-preview"],
    # Google Gemma
    "gemma-4-31b":          kbench.llms["google/gemma-4-31b"],
    "gemma-4-26b":          kbench.llms["google/gemma-4-26b-a4b"],
    # OpenAI
    "gpt-5-4":              kbench.llms["openai/gpt-5.4-2026-03-05"],
    "gpt-5-4-mini":         kbench.llms["openai/gpt-5.4-mini-2026-03-17"],
    # Qwen
    "qwen3-235b":           kbench.llms["qwen/qwen3-235b-a22b-instruct-2507"],
    "qwen3-coder-480b":     kbench.llms["qwen/qwen3-coder-480b-a35b-instruct"],
    # ZhipuAI
    "glm-5":                kbench.llms["zai/glm-5"],
}

In [ ]:
# Run t10_hallucination_abstention across all models
for name, model in ALL_MODELS.items():
    print(f'▶ t10_hallucination_abstention x {name}')
    t10_hallucination_abstention.run(model)


In [ ]:
%choose t10_hallucination_abstention